# CHARMS-adapted baseline — CSIRO Biomass

Adaptation of **CHARMS** (Jiang et al., ICML 2024 — *Tabular Insights, Visual Impacts*) to this dataset.

**Setting in the paper.** CHARMS assumes a training set with both an image and a set of *tabular attributes* per sample (e.g., pet type, vaccination status). At test time the tabular side is missing. It aligns image channels with tabular attributes via Optimal Transport and uses the alignment to inject tabular knowledge into the image model via an i2t auxiliary loss.

**This dataset.** We only have the 5 regression targets per image (no separate auxiliary tabular columns). We therefore treat the **5 target values themselves as the tabular attributes** during training. This is still faithful to the CHARMS idea: at test time no tabular info is available (we have to predict it), and the auxiliary channel-attribute alignment organizes the image feature space per-target. All 5 attributes are numerical → only the MSE branch of CHARMS's i2t loss is used (eq. 8, numerical part).

**What stays faithful to the paper:**
- Image feature extractor → `C` channels, K-Means clustered to `C'` groups (Sec. 4.2, 'Extract channel representation').
- Tabular encoder (small FT-style MLP) producing `D×E` attribute embeddings.
- Sample-wise cosine similarity matrices `S^I`, `S^T` → cost matrix `C_ij = ||S^I_i − S^T_j||²` (Sec. 4.2, 'Align two modalities').
- Sinkhorn OT → transfer matrix `T̂ ∈ R^{D×C'}`, un-clustered to `T ∈ R^{D×C}` (eq. 6).
- Loss `L = L_img + L_tab + L_i2t` (eq. 7–8).
- Cost matrix updated every 5 epochs (paper Sec. 4.3).

**Kaggle constraints respected:**
- No internet — ResNet is NOT used. A small CNN trained from scratch replaces the ResNet backbone. This is the honest move given no pretrained weights.
- Same paths (`/kaggle/input/competitions/csiro-biomass`, `/kaggle/working/submission.csv`).
- CPU-only friendly (small model, small batches, modest epoch count).
- Output format identical to baseline.

In [ ]:
import os
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.cluster import KMeans
from torch.utils.data import Dataset, DataLoader

TARGETS = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]
WEIGHTS = {
    "Dry_Green_g": 0.1,
    "Dry_Dead_g": 0.1,
    "Dry_Clover_g": 0.1,
    "GDM_g": 0.2,
    "Dry_Total_g": 0.5,
}

DATASET_DIR = "/kaggle/input/competitions/csiro-biomass"
TRAIN_CSV = os.path.join(DATASET_DIR, "train.csv")
TEST_CSV = os.path.join(DATASET_DIR, "test.csv")
TRAIN_IMG_DIR = os.path.join(DATASET_DIR, "train")
TEST_IMG_DIR = os.path.join(DATASET_DIR, "test")

print("train.csv exists:", os.path.exists(TRAIN_CSV))
print("test.csv exists:", os.path.exists(TEST_CSV))
print("train dir exists:", os.path.exists(TRAIN_IMG_DIR))
print("test dir exists:", os.path.exists(TEST_IMG_DIR))

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

In [ ]:
def load_train_data(csv_path):
    df = pd.read_csv(csv_path)
    df["image_id"] = df["sample_id"].str.split("__").str[0]
    df_wide = df.pivot_table(
        index=["image_id", "image_path"],
        columns="target_name",
        values="target",
    ).reset_index()
    y_values = df_wide[TARGETS].values.astype(np.float32)
    return df_wide, y_values


def prepare_submission(test_csv_path, predictions, image_ids):
    df_test = pd.read_csv(test_csv_path)
    pred_dict = {}
    for img_id, pred_vector in zip(image_ids, predictions):
        pred_dict[img_id] = {col: val for col, val in zip(TARGETS, pred_vector)}

    def get_pred(row):
        img_id = row["sample_id"].split("__")[0]
        target_name = row["target_name"]
        val = pred_dict.get(img_id, {}).get(target_name, 0.0)
        return max(0.0, float(val))

    df_test["target"] = df_test.apply(get_pred, axis=1)
    return df_test[["sample_id", "target"]]


def weighted_global_r2(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    w = np.array([WEIGHTS[t] for t in TARGETS], dtype=np.float64)
    yt = y_true.reshape(-1)
    yp = y_pred.reshape(-1)
    ww = np.repeat(w, y_true.shape[0])
    ybar = np.sum(ww * yt) / np.sum(ww)
    ss_res = np.sum(ww * (yt - yp) ** 2)
    ss_tot = np.sum(ww * (yt - ybar) ** 2) + 1e-12
    return float(1.0 - ss_res / ss_tot)


def rmse_per_target(y_true, y_pred):
    out = {}
    for i, t in enumerate(TARGETS):
        out[t] = float(np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i])))
    return out

In [ ]:
# CPU-friendly image size. The original baseline used 224x224 on each half.
# We keep two 128x128 halves (left/right) — the 2000x1000 image naturally splits in two.
IMG_HALF = 128

train_tfm = transforms.Compose([
    transforms.Resize((IMG_HALF, IMG_HALF)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
])

eval_tfm = transforms.Compose([
    transforms.Resize((IMG_HALF, IMG_HALF)),
    transforms.ToTensor(),
])


class BiomassDataset(Dataset):
    """Returns (img_left, img_right, tabular_attributes, target_vector, image_id).

    In the CHARMS framing the `tabular_attributes` is the auxiliary side used
    during training. For this dataset we use the same 5 target values — so the
    model learns image-channel / attribute alignment, and at test time the
    tabular side is simply not consumed (it is None).
    """

    def __init__(self, df, img_dir, y=None, transform=eval_tfm, has_tab=True):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.y = y
        self.transform = transform
        self.has_tab = has_tab

    def __len__(self):
        return len(self.df)

    def _load_halves(self, image_id):
        path = os.path.join(self.img_dir, f"{image_id}.jpg")
        try:
            img = Image.open(path).convert("RGB")
            w, h = img.size
            mid = w // 2
            img_left = img.crop((0, 0, mid, h))
            img_right = img.crop((mid, 0, w, h))
        except Exception as e:
            print(f"Error reading {path}: {e}")
            img_left = Image.new("RGB", (IMG_HALF, IMG_HALF))
            img_right = Image.new("RGB", (IMG_HALF, IMG_HALF))
        return self.transform(img_left), self.transform(img_right)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_id = row["image_id"]
        il, ir = self._load_halves(image_id)
        if self.y is not None:
            y = torch.from_numpy(self.y[idx]).float()
        else:
            y = torch.zeros(len(TARGETS), dtype=torch.float32)
        # tabular = the target values at train time
        tab = y.clone() if self.has_tab else torch.zeros_like(y)
        return il, ir, tab, y, image_id

In [ ]:
class SmallCNN(nn.Module):
    """CPU-friendly CNN that produces C channels before global pooling.

    Maps input Bx3xHxW -> Bx(C)xH'xW' (before pool) and Bx(C) after pool.
    Exposes `spatial_features` and `pooled_features`.
    """

    def __init__(self, out_channels=128):
        super().__init__()
        self.out_channels = out_channels
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 64
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 32
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 16
            nn.Conv2d(128, out_channels, 3, padding=1), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 8
        )
        self.pool = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        fmap = self.features(x)           # B x C x H' x W'
        pooled = self.pool(fmap).flatten(1)  # B x C
        return fmap, pooled


class ImageBranch(nn.Module):
    """Processes left and right halves separately, concatenates pooled features."""

    def __init__(self, per_half_channels=128, n_targets=5):
        super().__init__()
        self.left = SmallCNN(out_channels=per_half_channels)
        self.right = SmallCNN(out_channels=per_half_channels)
        self.channel_dim = 2 * per_half_channels  # C
        self.head = nn.Sequential(
            nn.Linear(self.channel_dim, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(128, n_targets),
        )

    def forward(self, xl, xr):
        _, pl = self.left(xl)
        _, pr = self.right(xr)
        pooled = torch.cat([pl, pr], dim=1)  # B x C
        yhat = self.head(pooled)
        return pooled, yhat


class TabularBranch(nn.Module):
    """Tiny 'FT-style' tabular encoder for numerical attributes.

    In the paper this is FT-Transformer; with 5 attributes and 357 rows a
    transformer is overkill and would hurt. We use a per-feature embedding +
    shared MLP. Outputs D attribute-embeddings of size E plus a task prediction.
    """

    def __init__(self, n_attrs=5, embed_dim=32):
        super().__init__()
        self.n_attrs = n_attrs
        self.embed_dim = embed_dim
        # per-attribute tokenizer: each scalar -> embed_dim vector.
        self.token_w = nn.Parameter(torch.randn(n_attrs, embed_dim) * 0.1)
        self.token_b = nn.Parameter(torch.zeros(n_attrs, embed_dim))
        # shared attribute transformer (small)
        self.attr_mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim), nn.ReLU(inplace=True),
            nn.Linear(embed_dim, embed_dim),
        )
        # task head operating on flattened attr embeddings
        self.head = nn.Sequential(
            nn.Linear(n_attrs * embed_dim, 64), nn.ReLU(inplace=True),
            nn.Linear(64, n_attrs),
        )

    def forward(self, tab):  # tab: B x D
        # per-attribute tokenization
        e = tab.unsqueeze(-1) * self.token_w.unsqueeze(0) + self.token_b.unsqueeze(0)  # B x D x E
        e = self.attr_mlp(e)  # B x D x E
        flat = e.flatten(1)
        yhat = self.head(flat)
        return e, yhat

In [ ]:
def sinkhorn(cost, a=None, b=None, reg=0.1, n_iter=50):
    """Entropy-regularised OT (Sinkhorn) on a DxC' cost matrix.

    Returns the transport plan T of shape DxC'.
    Pure PyTorch, works on CPU.
    """
    D, Cp = cost.shape
    if a is None:
        a = torch.full((D,), 1.0 / D, device=cost.device, dtype=cost.dtype)
    if b is None:
        b = torch.full((Cp,), 1.0 / Cp, device=cost.device, dtype=cost.dtype)
    K = torch.exp(-cost / reg)
    u = torch.ones_like(a)
    v = torch.ones_like(b)
    for _ in range(n_iter):
        u = a / (K @ v + 1e-9)
        v = b / (K.t() @ u + 1e-9)
    T = u.unsqueeze(1) * K * v.unsqueeze(0)
    return T


def sample_similarity(mat, eps=1e-8):
    """Return NxN cosine similarity for each 'item' along dim=1.

    mat shape: N x K. Returns K x N x N (one NxN matrix per 'item' view).
    Here we view it differently: we want, per image-channel i, an NxN similarity.
    We build it from an NxM matrix where column i is the activation for that
    channel / attribute across the N samples.
    """
    # mat: N x K  -> for each k, take column (N,) -> NxN sim matrix
    # similarity of sample a and sample b on channel k = x_ak * x_bk / (|x_ak||x_bk|)
    # With scalars this degenerates, so we treat mat as N items with K-dim feature
    # We use vector per-sample and compute NxN cosine over the full feature.
    x = mat / (mat.norm(dim=1, keepdim=True) + eps)
    return x @ x.t()


def per_channel_similarity(activ, eps=1e-8):
    """activ: N x K (K channels or attrs). Return K x N x N sample similarity per channel/attr.

    We treat each channel's scalar activation across N samples as a 1-D signal
    and build the outer-product correlation:
        S_k[a,b] = a_k * b_k / (|a_k| * |b_k|) -> sign-only; too coarse.

    Instead, following the paper, each channel i is represented by an N-vector
    of activations across samples. The 'sample cosine similarity on channel i'
    is then the normalized outer product of that vector with itself — a rank-1
    matrix. That's informative enough to contrast channels.
    """
    # activ: N x K  ->  K x N
    v = activ.t()  # K x N
    # Normalise each channel's N-vector
    v_n = v / (v.norm(dim=1, keepdim=True) + eps)
    # Rank-1 similarity matrix per channel: K x N x N
    S = torch.einsum("ka,kb->kab", v_n, v_n)
    return S


def build_cost_matrix(S_T, S_I):
    """S_T: D x N x N, S_I: Cp x N x N.  Return DxCp cost matrix.

    cost[j,i] = ||S^T_j - S^I_i||_F^2
    """
    D = S_T.size(0)
    Cp = S_I.size(0)
    # flatten and use pairwise squared distance
    T_flat = S_T.reshape(D, -1)
    I_flat = S_I.reshape(Cp, -1)
    # squared distance between rows of T_flat and I_flat
    d = (T_flat ** 2).sum(1, keepdim=True) + (I_flat ** 2).sum(1).unsqueeze(0) - 2 * T_flat @ I_flat.t()
    d = d.clamp(min=0.0)
    return d  # D x Cp


def kmeans_cluster_channels(channel_vecs, n_clusters, seed=SEED):
    """channel_vecs: C x N (each row is a channel's activation across N samples).

    Returns cluster_labels (C,) and cluster_centers (n_clusters x N).
    Uses sklearn KMeans to avoid custom impl.
    """
    X = channel_vecs.detach().cpu().numpy()
    km = KMeans(n_clusters=n_clusters, n_init=5, random_state=seed)
    labels = km.fit_predict(X)
    return labels, km.cluster_centers_

In [ ]:
train_df, y_all = load_train_data(TRAIN_CSV)
print("Train images:", len(train_df))
print("Target shape:", y_all.shape)
print(train_df.head())

# Normalise targets for loss stability (scale back at inference).
# Store scale so we can invert predictions.
y_mean = y_all.mean(axis=0, keepdims=True)
y_std = y_all.std(axis=0, keepdims=True) + 1e-6
y_all_norm = (y_all - y_mean) / y_std
print("y_mean:", y_mean.flatten())
print("y_std :", y_std.flatten())

train_idx, val_idx = train_test_split(
    np.arange(len(train_df)), test_size=0.2, random_state=SEED
)

train_subset = train_df.iloc[train_idx].reset_index(drop=True)
val_subset = train_df.iloc[val_idx].reset_index(drop=True)
y_tr_norm = y_all_norm[train_idx]
y_val_norm = y_all_norm[val_idx]
y_val_raw = y_all[val_idx]

print("Train split:", len(train_subset), "Val split:", len(val_subset))

BATCH_SIZE = 32
NUM_WORKERS = 2

train_ds = BiomassDataset(train_subset, TRAIN_IMG_DIR, y_tr_norm, transform=train_tfm, has_tab=True)
val_ds = BiomassDataset(val_subset, TRAIN_IMG_DIR, y_val_norm, transform=eval_tfm, has_tab=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print("Batches per epoch:", len(train_loader))

In [ ]:
PER_HALF_CHANNELS = 64      # C / 2
EMBED_DIM = 16              # E
N_TARGETS = len(TARGETS)    # D = 5
N_CLUSTERS = 16             # C'  (paper uses 40; we go smaller — fewer channels)

img_model = ImageBranch(per_half_channels=PER_HALF_CHANNELS, n_targets=N_TARGETS).to(device)
tab_model = TabularBranch(n_attrs=N_TARGETS, embed_dim=EMBED_DIM).to(device)

# Channel-to-attribute linear heads (one per numerical attribute).
# Each head maps the (masked/weighted) image embedding (C-dim) -> 1 scalar.
C_TOTAL = img_model.channel_dim
i2t_heads = nn.ModuleList([
    nn.Linear(C_TOTAL, 1) for _ in range(N_TARGETS)
]).to(device)

params = list(img_model.parameters()) + list(tab_model.parameters()) + list(i2t_heads.parameters())
optimiser = torch.optim.AdamW(params, lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=30)

print("Image branch channels (C):", C_TOTAL)
print("N clusters (C'):", N_CLUSTERS)
print("N attributes (D):", N_TARGETS)

n_params = sum(p.numel() for p in params)
print(f"Total trainable parameters: {n_params:,}")

In [ ]:
@torch.no_grad()
def compute_transfer_matrix(img_model, tab_model, loader, n_clusters, device):
    """Walk the (training) loader once, collect per-sample channel pooled
    activations and per-attribute embeddings, cluster channels, compute cost,
    Sinkhorn-align, and return T ∈ R^{D x C}.
    """
    img_model.eval()
    tab_model.eval()

    pooled_list = []
    attr_list = []  # each element: D x E (per sample)

    for il, ir, tab, _y, _ in loader:
        il = il.to(device); ir = ir.to(device); tab = tab.to(device)
        pooled, _ = img_model(il, ir)       # B x C
        e, _ = tab_model(tab)               # B x D x E
        pooled_list.append(pooled.cpu())
        attr_list.append(e.cpu())

    pooled = torch.cat(pooled_list, dim=0)  # N x C
    attrs = torch.cat(attr_list, dim=0)     # N x D x E
    N, C = pooled.shape
    D = attrs.size(1)

    # ---- Channel clustering (paper Sec. 4.2) ----
    # channel_vecs: C x N  (each row = one channel's activation across the N train samples)
    channel_vecs = pooled.t().contiguous()
    if n_clusters >= C:
        labels = np.arange(C)
        n_clusters_eff = C
    else:
        labels, _ = kmeans_cluster_channels(channel_vecs, n_clusters)
        n_clusters_eff = n_clusters

    # Build clustered channel vectors: average channels within cluster
    clustered = torch.zeros(n_clusters_eff, N)
    counts = torch.zeros(n_clusters_eff)
    for c_idx, lab in enumerate(labels):
        clustered[lab] += channel_vecs[c_idx]
        counts[lab] += 1
    clustered = clustered / counts.clamp(min=1).unsqueeze(1)

    # ---- Per-channel (cluster) and per-attribute sample similarity ----
    # For image: use clustered NxN similarity: each cluster is an N-vector
    S_I = per_channel_similarity(clustered.t())    # needs N x Cp shape -> transpose back
    # For tab: each attribute is a D-slice; per sample we have a E-dim embedding
    # We build each attribute's 'N-signature' by taking L2 norm of embedding per sample.
    attr_sig = attrs.norm(dim=-1)                   # N x D
    S_T = per_channel_similarity(attr_sig)          # D x N x N

    # ---- Cost matrix & Sinkhorn ----
    cost = build_cost_matrix(S_T, S_I)               # D x Cp
    T_hat = sinkhorn(cost, reg=0.1, n_iter=100)      # D x Cp

    # Map clustered transfer matrix back to per-channel T (paper: 'restore')
    T_full = torch.zeros(D, C)
    for c_idx, lab in enumerate(labels):
        T_full[:, c_idx] = T_hat[:, lab]

    # Row-wise normalisation so each attribute's weights sum to ~1
    T_full = T_full / (T_full.sum(dim=1, keepdim=True) + 1e-8)
    return T_full.to(device), labels, T_hat.cpu()


# Sanity check before training
T_init, _, _ = compute_transfer_matrix(img_model, tab_model, train_loader, N_CLUSTERS, device)
print("Initial transfer matrix T shape:", tuple(T_init.shape))
print("Row sums (should be ~1):", T_init.sum(dim=1).cpu().numpy())

In [ ]:
N_EPOCHS = 30
OT_UPDATE_EVERY = 5   # paper: 'We update cost matrix every 5 epochs'
LAMBDA_I2T = 0.3      # weight on i2t auxiliary loss
LAMBDA_TAB = 0.5      # weight on tabular-branch supervised loss


def run_validation(img_model, val_loader, device, y_mean, y_std, y_val_raw):
    img_model.eval()
    preds = []
    with torch.no_grad():
        for il, ir, _tab, _y, _ in val_loader:
            il = il.to(device); ir = ir.to(device)
            _, yhat = img_model(il, ir)
            preds.append(yhat.cpu().numpy())
    preds_norm = np.concatenate(preds, axis=0)
    preds_raw = preds_norm * y_std + y_mean
    preds_raw = np.clip(preds_raw, 0.0, None)
    return (
        weighted_global_r2(y_val_raw, preds_raw),
        r2_score(y_val_raw, preds_raw, multioutput="uniform_average"),
        rmse_per_target(y_val_raw, preds_raw),
    )


T_matrix = T_init.detach()
best_wr2 = -1e9
best_state = None

for epoch in range(1, N_EPOCHS + 1):
    img_model.train(); tab_model.train(); i2t_heads.train()
    epoch_total = 0.0
    epoch_img = 0.0
    epoch_tab = 0.0
    epoch_i2t = 0.0
    n_batches = 0

    for il, ir, tab, y, _ in train_loader:
        il = il.to(device); ir = ir.to(device); tab = tab.to(device); y = y.to(device)

        pooled, y_img = img_model(il, ir)   # pooled: B x C,  y_img: B x D
        _, y_tab = tab_model(tab)            # y_tab: B x D

        # --- Supervised losses ---
        loss_img = F.mse_loss(y_img, y)
        loss_tab = F.mse_loss(y_tab, y)

        # --- i2t auxiliary (paper eq. 8, numerical branch) ---
        #   For each attribute p: take T[p] (length C) -> weight channels -> predict tab[p]
        loss_i2t = 0.0
        for p in range(N_TARGETS):
            w = T_matrix[p].detach()                       # C,
            weighted = pooled * w.unsqueeze(0)             # B x C
            pred_p = i2t_heads[p](weighted).squeeze(-1)    # B,
            loss_i2t = loss_i2t + F.mse_loss(pred_p, tab[:, p])
        loss_i2t = loss_i2t / N_TARGETS

        loss = loss_img + LAMBDA_TAB * loss_tab + LAMBDA_I2T * loss_i2t

        optimiser.zero_grad()
        loss.backward()
        optimiser.step()

        epoch_total += float(loss.item())
        epoch_img += float(loss_img.item())
        epoch_tab += float(loss_tab.item())
        epoch_i2t += float(loss_i2t.item() if torch.is_tensor(loss_i2t) else loss_i2t)
        n_batches += 1

    scheduler.step()

    # Refresh transfer matrix every OT_UPDATE_EVERY epochs
    if epoch % OT_UPDATE_EVERY == 0 or epoch == 1:
        T_matrix, _, _ = compute_transfer_matrix(
            img_model, tab_model, train_loader, N_CLUSTERS, device
        )
        T_matrix = T_matrix.detach()

    if epoch % 2 == 0 or epoch == 1 or epoch == N_EPOCHS:
        wr2, r2_u, rmse = run_validation(img_model, val_loader, device, y_mean, y_std, y_val_raw)
        print(
            f"ep {epoch:02d} | total {epoch_total / n_batches:.4f} "
            f"| img {epoch_img / n_batches:.4f} "
            f"| tab {epoch_tab / n_batches:.4f} "
            f"| i2t {epoch_i2t / n_batches:.4f} "
            f"| val wR² {wr2:.4f} | val R² {r2_u:.4f}"
        )
        if wr2 > best_wr2:
            best_wr2 = wr2
            best_state = {k: v.detach().cpu().clone() for k, v in img_model.state_dict().items()}
            print(f"   ↳ new best (wR² = {best_wr2:.4f})")

print("\nTraining done. Best val wR²:", best_wr2)
if best_state is not None:
    img_model.load_state_dict(best_state)
    print("Restored best image-model weights.")

In [ ]:
wr2, r2_u, rmse = run_validation(img_model, val_loader, device, y_mean, y_std, y_val_raw)
print("Final validation weighted global R²:", wr2)
print("Final validation uniform R² :", r2_u)
print("Final validation RMSE per target:")
print(rmse)

In [ ]:
# Re-train briefly on ALL training data for the final predictor (paper keeps the
# same loss formulation; we just drop the val split). A few extra epochs with
# the best-so-far weights as initialisation.

full_ds = BiomassDataset(train_df.reset_index(drop=True), TRAIN_IMG_DIR, y_all_norm, transform=train_tfm, has_tab=True)
full_loader = DataLoader(full_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

optimiser_ft = torch.optim.AdamW(
    list(img_model.parameters()) + list(tab_model.parameters()) + list(i2t_heads.parameters()),
    lr=5e-4, weight_decay=1e-4,
)

FT_EPOCHS = 8

# Refresh T on full data once
T_matrix, _, _ = compute_transfer_matrix(img_model, tab_model, full_loader, N_CLUSTERS, device)
T_matrix = T_matrix.detach()

for epoch in range(1, FT_EPOCHS + 1):
    img_model.train(); tab_model.train(); i2t_heads.train()
    total = 0.0
    for il, ir, tab, y, _ in full_loader:
        il = il.to(device); ir = ir.to(device); tab = tab.to(device); y = y.to(device)
        pooled, y_img = img_model(il, ir)
        _, y_tab = tab_model(tab)
        loss_img = F.mse_loss(y_img, y)
        loss_tab = F.mse_loss(y_tab, y)
        loss_i2t = 0.0
        for p in range(N_TARGETS):
            w = T_matrix[p].detach()
            pred_p = i2t_heads[p](pooled * w.unsqueeze(0)).squeeze(-1)
            loss_i2t = loss_i2t + F.mse_loss(pred_p, tab[:, p])
        loss_i2t = loss_i2t / N_TARGETS
        loss = loss_img + LAMBDA_TAB * loss_tab + LAMBDA_I2T * loss_i2t
        optimiser_ft.zero_grad(); loss.backward(); optimiser_ft.step()
        total += float(loss.item())
    if epoch % OT_UPDATE_EVERY == 0:
        T_matrix, _, _ = compute_transfer_matrix(img_model, tab_model, full_loader, N_CLUSTERS, device)
        T_matrix = T_matrix.detach()
    print(f"FT ep {epoch:02d} | avg loss {total / len(full_loader):.4f}")

print("Final model trained on all training data.")

In [ ]:
df_test = pd.read_csv(TEST_CSV)
df_test["image_id"] = df_test["sample_id"].str.split("__").str[0]
df_test_unique = df_test.drop_duplicates("image_id").copy().reset_index(drop=True)
print("Test images:", len(df_test_unique))
print(df_test_unique.head())

# IMPORTANT: at test time we do NOT use tab info — that's the whole point of CHARMS.
test_ds = BiomassDataset(df_test_unique, TEST_IMG_DIR, y=None, transform=eval_tfm, has_tab=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

img_model.eval()
test_preds_norm = []
test_image_ids = []
with torch.no_grad():
    for il, ir, _tab, _y, ids in test_loader:
        il = il.to(device); ir = ir.to(device)
        _, yhat = img_model(il, ir)
        test_preds_norm.append(yhat.cpu().numpy())
        test_image_ids.extend(list(ids))

test_preds_norm = np.concatenate(test_preds_norm, axis=0)
test_preds = test_preds_norm * y_std + y_mean
test_preds = np.clip(test_preds, 0.0, None)
print("test_preds shape:", test_preds.shape)

submission = prepare_submission(TEST_CSV, test_preds, test_image_ids)
print(submission.head())

submission.to_csv("/kaggle/working/submission.csv", index=False)
print("Saved submission to /kaggle/working/submission.csv")
print("Working dir files:", os.listdir("/kaggle/working"))